In [0]:
%pip install openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 35.1 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.10.0
    Not uninstalling typing-extensions at /databricks/python3/lib/python3.11/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-7be62f47-33e2-471b-a953-91f33817729e
    Can't uninstall 'typing_extensions'. No files were found to uninstall.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%pip install openai
from openai import OpenAI

import json
from pyspark.sql.functions import *

DATABRICKS_TOKEN = (
    dbutils.notebook
    .entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .apiToken()
    .get()
)

_ws_raw = spark.conf.get("spark.databricks.workspaceUrl")
_ws_url = _ws_raw.replace("https://", "").replace("http://", "").strip("/")

client = OpenAI(
    api_key=DATABRICKS_TOKEN,
    base_url=f"https://{_ws_url}/serving-endpoints"
)

MODEL_NAME = "databricks-gpt-oss-20b"

# ============================================================
# STEP 1: Build per-customer return profile
# ============================================================


# Join returns back to sales to compute return timing
# ------------------------------------------------------------------

return_profile_df = spark.sql("""
    SELECT
        customer_id,
        COUNT(*) AS total_returns,
        SUM(r.refund_amount) AS total_return_value,
        MAX(r.date) AS latest_return_date,
        COUNT(*) FILTER (
            WHERE r.date >= date_sub(current_date(), 30)
        ) AS last_30_day_returns
    FROM globalmart.gold.fact_returns r
    GROUP BY customer_id
""")

# ------------------------------------------------------------------
# Customer-level sales aggregation
# ------------------------------------------------------------------

sales_profile_df = spark.sql("""
    SELECT
        customer_id,
        COUNT(DISTINCT order_id) AS total_orders,
        SUM(sales_amount) AS total_sales_value
    FROM globalmart.gold.fact_transactions
    GROUP BY customer_id
""")

profile_df = (
    return_profile_df
    .join(sales_profile_df, "customer_id", "left")
    .fillna(0)
    .withColumn(
        "return_to_order_ratio",
        col("total_returns") / when(col("total_orders") == 0, 1).otherwise(col("total_orders"))
    )
)


# ============================================================
# STEP 2: Apply anomaly rules and compute score (0–100)
# ============================================================

# ------------------------------------------------------------------
# Rule-based anomaly scoring
# ------------------------------------------------------------------

scored_df = (
    profile_df

    # Rule 1: High return volume
    .withColumn(
        "rule_high_volume",
        when(col("total_returns") >= 5, 25).otherwise(0)
    )

    # Rule 2: High refund value
    .withColumn(
        "rule_high_value",
        when(col("total_return_value") >= 25000, 20).otherwise(0)
    )

    # Rule 3: High return-to-order ratio
    .withColumn(
        "rule_high_ratio",
        when(col("return_to_order_ratio") >= 0.5, 25).otherwise(0)
    )

    # Rule 4: Recent spike in returns
    .withColumn(
        "rule_recent_spike",
        when(col("last_30_day_returns") >= 3, 20).otherwise(0)
    )

    # Rule 5: Refunds exceed purchases
    .withColumn(
        "rule_refund_dominance",
        when(col("total_return_value") > col("total_sales_value"), 10).otherwise(0)
    )

    # Total anomaly score
    .withColumn(
        "anomaly_score",
        col("rule_high_volume")
        + col("rule_high_value")
        + col("rule_high_ratio")
        + col("rule_recent_spike")
        + col("rule_refund_dominance")
    )

    # Human-readable rule list
    .withColumn(
        "violated_rules",
        concat_ws(", ",
            when(col("rule_high_volume") > 0, lit("High return volume")),
            when(col("rule_high_value") > 0, lit("High refund value")),
            when(col("rule_high_ratio") > 0, lit("High return-to-order ratio")),
            when(col("rule_recent_spike") > 0, lit("Recent return spike")),
            when(col("rule_refund_dominance") > 0, lit("Refunds exceed purchases"))
        )
    )
)
display(scored_df)


# ============================================================
# STEP 3: Filter customers above risk threshold
# ============================================================

FLAG_THRESHOLD = 40  # Investigate above 50/100

flagged_df = scored_df.filter(col("anomaly_score") >= FLAG_THRESHOLD)

flagged_rows = flagged_df.collect()

def generate_investigation_brief(row):

    prompt = f"""
You are an AI assistant supporting a retail fraud investigations team.

Customer return profile:
- Customer ID: {row['customer_id']}
- Total returns: {row['total_returns']}
- Total refund value: {row['total_return_value']}
- Return-to-order ratio: {round(row['return_to_order_ratio'], 2)}
- Anomaly score: {row['anomaly_score']}
- Rules violated: {row['violated_rules']}

Write a 3–4 sentence investigation brief that:
1. Explains what specific data patterns make this case higher risk
2. Mentions plausible innocent explanations
3. Recommends what investigators should review first

Use professional, cautious language. Do not assume fraud.
"""

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}]
    )

    for block in response.choices[0].message.content:
        if block.get("type") == "text":
            return block.get("text")

    return "Investigation brief unavailable."

# ============================================================
# STEP 5: Write flagged customers to Gold layer
# ============================================================
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType,
    DoubleType, TimestampType
)
output_rows = []
schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("total_returns", IntegerType(), True),
    StructField("total_return_value", DoubleType(), True),
    StructField("anomaly_score", IntegerType(), True),
    StructField("violated_rules", StringType(), True),
    StructField("investigation_brief", StringType(), True),
    StructField("generated_at", TimestampType(), True)
])

for row in flagged_rows:
    brief = generate_investigation_brief(row)

    output_rows.append((
        row["customer_id"],
        row["total_returns"],
        float(row["total_return_value"]),
        row["anomaly_score"],
        row["violated_rules"],
        brief
    ))

if len(output_rows) == 0:
    final_df = spark.createDataFrame([], schema)
else:
    final_df = (
        spark.createDataFrame(
            final_rows,
            [
                "customer_id",
                "total_returns",
                "total_return_value",
                "anomaly_score",
                "violated_rules",
                "investigation_brief"
            ]
        )
        .withColumn("generated_at", current_timestamp())
    )
display(final_df)
final_df.write.mode("overwrite") \
    .saveAsTable("globalmart.gold.flagged_return_customers")



### Example Exploratory Notebook

Use this notebook to explore the data generated by the pipeline in your preferred programming language.

**Note**: This notebook is not executed as part of the pipeline.

In [0]:
spark.sql("USE CATALOG `globalmart`")
spark.sql("USE SCHEMA `gold`")

In [0]:
display(spark.sql("SELECT * FROM samples.wanderbricks.users"))